# Day 2 — Notebook 06
# RAG Evaluation, Hallucination Testing & Governance — Stable Azure Version

**Hands-on outcome:** Evaluate the RAG system

**Metrics:** Retrieval Hit, Citation Presence, Context Relevance, Faithfulness, Answer Relevance, Factual Correctness, Unsupported Claims.

> “A production RAG system needs a repeatable evaluation contract — not one good-looking demo answer.”

In [1]:
from pathlib import Path
import os, json, re
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display
import chromadb

ROOT = Path(".")
ARTIFACT_DIR = ROOT / "artifacts"
load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

client = OpenAI(base_url=endpoint, api_key=api_key)

chroma_client = chromadb.PersistentClient(path=str(ARTIFACT_DIR / "chroma_policy_db"))
collection = chroma_client.get_collection("policy_section")

TESTSET = json.loads(
    (ROOT / "data" / "evaluation" / "rag_evaluation_testset.json").read_text(encoding="utf-8")
)

REFERENCE_ANSWERS = {
    "Q01":"For Gold PPO, prior authorization is required beginning with the 11th physical therapy visit.",
    "Q02":"For Silver HMO, prior authorization is required beginning with the 7th physical therapy visit.",
    "Q03":"Emergency department advanced imaging does not require prior authorization.",
    "Q04":"Gold PPO covers up to 20 chiropractic visits per benefit year when medically necessary.",
    "Q05":"Silver HMO covers up to 12 chiropractic visits per benefit year when medically necessary.",
    "Q06":"A provider appeal should generally be submitted within 60 calendar days from the denial notice.",
    "Q07":"The standard clean-claim filing limit for participating providers is 180 calendar days from date of service unless an agreement states otherwise.",
    "Q08":"No. Prior authorization does not guarantee claim payment.",
    "Q09":"An advanced imaging request should include clinical indication, relevant history, prior conservative treatment where applicable, relevant prior imaging, and the requested study.",
    "Q10":"Insufficient information. The supplied Gold PPO policy does not state an annual acupuncture visit limit."
}

display(pd.DataFrame([
    {"ID":x["id"],"Question":x["question"],"Expected Source":x.get("expected_doc_id"),
     "Expected Section":x.get("expected_section"),"Plan":x.get("plan_type"),
     "Reference Answer":REFERENCE_ANSWERS[x["id"]]}
    for x in TESTSET
]))

NotFoundError: Collection [policy_section] does not exist

> **Takeaway:** The test set is the evaluation contract. Keep the questions fixed while changing retrieval or prompting.

In [ ]:
def infer_plan_filter(question):
    q = question.lower()
    if "gold ppo" in q: return "Gold PPO"
    if "silver hmo" in q: return "Silver HMO"
    return None

def retrieve(question, top_k=4, use_plan_filter=True):
    plan = infer_plan_filter(question) if use_plan_filter else None
    qvec = client.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    raw = collection.query(
        query_embeddings=[qvec],
        n_results=top_k,
        where={"plan_type":plan} if plan else None,
        include=["documents","metadatas","distances"]
    )
    chunks = []
    for rank,(text,meta,distance) in enumerate(
        zip(raw["documents"][0],raw["metadatas"][0],raw["distances"][0]), start=1
    ):
        chunks.append({
            "rank":rank,"text":text,"metadata":meta,"distance":distance,
            "similarity":1-distance,
            "citation":f"{meta['doc_id']} | {meta['section']} | p.{meta['page']}"
        })
    return chunks

def assemble_context(chunks):
    return "\n\n".join(
        f"[SOURCE {i}: {c['citation']}]\n{c['text']}"
        for i,c in enumerate(chunks,start=1)
    )

SYSTEM = """
You are a healthcare payer policy assistant.
Use only the provided context.
If evidence is insufficient, say exactly: Insufficient information.
Do not infer missing benefit rules.
Cite supporting source labels exactly as [SOURCE n].
Keep the answer concise.
"""

def answer_question(question, chunks, system_prompt=SYSTEM):
    return client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":f"CONTEXT:\n{assemble_context(chunks)}\n\nQUESTION:\n{question}"}
        ],
        temperature=0
    )

## Metric definitions

In [ ]:
display(pd.DataFrame([
    {"Metric":"Retrieval Hit","Layer":"Retriever","Meaning":"Expected policy section appeared","Scale":"True/False"},
    {"Metric":"Citation Presence","Layer":"Governance","Meaning":"Answer exposed source citation","Scale":"True/False"},
    {"Metric":"Context Relevance","Layer":"Retriever","Meaning":"Retrieved chunks are relevant","Scale":"0–5"},
    {"Metric":"Faithfulness","Layer":"Generator","Meaning":"Claims are supported by context","Scale":"0–5"},
    {"Metric":"Answer Relevance","Layer":"Generator","Meaning":"Answer addresses the question","Scale":"0–5"},
    {"Metric":"Factual Correctness","Layer":"Generator","Meaning":"Answer agrees with reference","Scale":"0–5"},
    {"Metric":"Unsupported Claims","Layer":"Responsible AI","Meaning":"Claims not supported by evidence","Scale":"Count"}
]))

In [ ]:
def retrieval_hit(chunks, expected_doc_id, expected_section):
    if expected_doc_id is None: return None
    return any(
        c["metadata"]["doc_id"] == expected_doc_id and
        c["metadata"]["section"].upper() == expected_section.upper()
        for c in chunks
    )

def citation_present(answer):
    return "[source" in answer.lower()

EVALUATOR_SYSTEM = """
You are a strict healthcare RAG evaluator.
Return ONLY valid JSON:
{
  "context_relevance": 0,
  "faithfulness": 0,
  "answer_relevance": 0,
  "factual_correctness": 0,
  "unsupported_claims": [],
  "reason": ""
}
Score each numeric metric from 0 to 5.
Evaluate only from the supplied question, retrieved context, generated answer and reference answer.
"""

def evaluate_answer(question, answer, retrieved_chunks, reference_answer):
    prompt = f"""
QUESTION:
{question}

RETRIEVED CONTEXT:
{chr(10).join(c["text"] for c in retrieved_chunks)}

GENERATED ANSWER:
{answer}

REFERENCE ANSWER:
{reference_answer}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":EVALUATOR_SYSTEM},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )
    text = response.choices[0].message.content.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.I|re.S)
    return json.loads(text)

## Run the complete evaluation set

In [ ]:
evaluation_results = []
trace_rows = []

for case in TESTSET:
    chunks = retrieve(case["question"], top_k=4, use_plan_filter=True)
    response = answer_question(case["question"], chunks)
    answer = response.choices[0].message.content
    scores = evaluate_answer(
        case["question"], answer, chunks, REFERENCE_ANSWERS[case["id"]]
    )

    row = {
        "ID":case["id"],
        "Retrieval Hit":retrieval_hit(chunks, case.get("expected_doc_id"), case.get("expected_section")),
        "Citation Present":citation_present(answer),
        "Context Relevance":scores["context_relevance"],
        "Faithfulness":scores["faithfulness"],
        "Answer Relevance":scores["answer_relevance"],
        "Factual Correctness":scores["factual_correctness"],
        "Unsupported Claims":len(scores.get("unsupported_claims", [])),
        "Answer":answer,
        "Evaluation Reason":scores.get("reason","")
    }
    evaluation_results.append(row)
    trace_rows.append({
        "id":case["id"],"question":case["question"],"answer":answer,
        "retrieved_sources":[c["citation"] for c in chunks],
        "retrieved_contexts":[c["text"] for c in chunks],
        "evaluation":row,
        "usage":{
            "prompt_tokens":response.usage.prompt_tokens,
            "completion_tokens":response.usage.completion_tokens,
            "total_tokens":response.usage.total_tokens
        }
    })

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df[
    ["ID","Retrieval Hit","Citation Present","Context Relevance","Faithfulness",
     "Answer Relevance","Factual Correctness","Unsupported Claims"]
])

## Management-friendly scorecard

In [ ]:
scorecard = pd.DataFrame([
    {"Metric":"Retrieval Hit Rate","Score":round(evaluation_df["Retrieval Hit"].dropna().astype(float).mean(),3),"Scale":"0–1"},
    {"Metric":"Citation Rate","Score":round(evaluation_df["Citation Present"].astype(float).mean(),3),"Scale":"0–1"},
    {"Metric":"Context Relevance","Score":round(evaluation_df["Context Relevance"].mean(),2),"Scale":"0–5"},
    {"Metric":"Faithfulness","Score":round(evaluation_df["Faithfulness"].mean(),2),"Scale":"0–5"},
    {"Metric":"Answer Relevance","Score":round(evaluation_df["Answer Relevance"].mean(),2),"Scale":"0–5"},
    {"Metric":"Factual Correctness","Score":round(evaluation_df["Factual Correctness"].mean(),2),"Scale":"0–5"},
    {"Metric":"Unsupported Claims","Score":int(evaluation_df["Unsupported Claims"].sum()),"Scale":"Count"}
])
display(scorecard)

## Hallucination experiment — irresponsible vs responsible output

In [ ]:
hall_q = "What is the annual acupuncture visit limit under the Gold PPO plan?"
hall_chunks = retrieve(hall_q, top_k=4, use_plan_filter=True)

IRRESPONSIBLE_SYSTEM = """
You are a confident healthcare benefits assistant.
If the supplied context does not contain an exact value,
infer a plausible industry-standard value.
Do not mention uncertainty.
"""

bad_answer = answer_question(hall_q, hall_chunks, IRRESPONSIBLE_SYSTEM).choices[0].message.content
good_answer = answer_question(hall_q, hall_chunks, SYSTEM).choices[0].message.content

bad_scores = evaluate_answer(hall_q, bad_answer, hall_chunks, REFERENCE_ANSWERS["Q10"])
good_scores = evaluate_answer(hall_q, good_answer, hall_chunks, REFERENCE_ANSWERS["Q10"])

display(pd.DataFrame([
    {
        "Behavior":"Irresponsible",
        "Answer":bad_answer,
        "Faithfulness":bad_scores["faithfulness"],
        "Factual Correctness":bad_scores["factual_correctness"],
        "Unsupported Claims":len(bad_scores.get("unsupported_claims",[])),
        "Responsible Outcome?":"No"
    },
    {
        "Behavior":"Responsible",
        "Answer":good_answer,
        "Faithfulness":good_scores["faithfulness"],
        "Factual Correctness":good_scores["factual_correctness"],
        "Unsupported Claims":len(good_scores.get("unsupported_claims",[])),
        "Responsible Outcome?":"Yes"
    }
]))

> **Takeaway:** A hallucinated answer can be relevant but still unfaithful. Relevance alone is not enough.

## Baseline vs improved retrieval

In [ ]:
def retrieval_run(use_plan_filter):
    rows=[]
    for case in TESTSET:
        if case.get("expected_doc_id") is None:
            continue
        chunks = retrieve(case["question"], top_k=4, use_plan_filter=use_plan_filter)
        rows.append({
            "ID":case["id"],
            "Retrieval Hit":retrieval_hit(chunks,case.get("expected_doc_id"),case.get("expected_section")),
            "Top Document":chunks[0]["metadata"]["doc_id"],
            "Top Section":chunks[0]["metadata"]["section"]
        })
    return pd.DataFrame(rows)

baseline = retrieval_run(False)
improved = retrieval_run(True)

comparison = baseline.merge(improved,on="ID",suffixes=("_Baseline","_Improved"))
display(comparison)

display(pd.DataFrame([
    {"Run":"Baseline - semantic only","Retrieval Hit Rate":round(baseline["Retrieval Hit"].astype(float).mean(),3)},
    {"Run":"Improved - plan-aware filter","Retrieval Hit Rate":round(improved["Retrieval Hit"].astype(float).mean(),3)}
]))

> **Takeaway:** Measure → diagnose → change one control → rerun the same test set.

## Governance control map

In [ ]:
display(pd.DataFrame([
    {"Risk":"Wrong plan / version","Technical Control":"Metadata filtering","Evidence to Retain":"Applied filters + source metadata"},
    {"Risk":"Hallucinated fact","Technical Control":"Evidence-only prompt + faithfulness check","Evidence to Retain":"Context + answer + score"},
    {"Risk":"Poor ranking","Technical Control":"Similarity ranking + reranking","Evidence to Retain":"Initial rank + final rank"},
    {"Risk":"Missing evidence","Technical Control":"Insufficient-information response","Evidence to Retain":"Context + evidence gap"},
    {"Risk":"Untraceable answer","Technical Control":"Citation metadata","Evidence to Retain":"Document ID + section + page"},
    {"Risk":"High-impact automated decision","Technical Control":"Human review / escalation","Evidence to Retain":"Escalation reason + reviewer decision"}
]))

## Save audit trace

In [ ]:
trace_path = ARTIFACT_DIR / "rag_evaluation_stable_trace.jsonl"
with open(trace_path,"w",encoding="utf-8") as f:
    for trace in trace_rows:
        f.write(json.dumps(trace,ensure_ascii=False,default=str)+"\n")

display(pd.DataFrame([{
    "Artifact":"RAG Evaluation Trace",
    "Path":str(trace_path),
    "Contains":"Question, retrieved evidence, answer, metrics and token usage"
}]))

# Final Day 2 outcome

**Documents → chunking → embeddings → similarity → ranking → reranking → grounded answer → hallucination control → evaluation → improvement → governance trace**

**Optional theory reference:** Frameworks such as Ragas can automate similar evaluation patterns, but this hands-on keeps the evaluation logic transparent and stable.